# Traceprop review runs (items 1, 2, 8)

Produces the result files `docs/mlsys/REVIEW_SCAFFOLD.md` needs. Run on an **A100** runtime;
job 4 is CPU-only and runs fine in the same A100 session. **Item 9 (rank agreement) is excluded**:
it needs a score-matrix step not yet wired.

Each run writes to `experiments/results/`; the last cell copies them to Drive. Download these back
into your local `experiments/results/`, then tell the assistant:

- `exp36_memory_storage_pythia1b.json`
- `exp35_inlineprecond_overhead_hf_pythia7b_split.json`
- `exp35_overhead_1b_k8.json`, `exp35_overhead_1b_k32.json`, `exp35_overhead_1b_k64.json`
- `exp35_ksweep_seed0.npz` ... `exp35_ksweep_seed4.npz`

`exp36` is a new script not yet run on GPU; if it errors, send the traceback.

In [1]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only'
print('GPU:', name)
if 'A100' not in name:
    print('WARNING: jobs 1-3 expect an A100 (40GB); results on a different GPU are not comparable '
          'to the paper. Job 4 (quality sweep) is CPU-only and is fine on any runtime.')

# Install matches the working stage2 notebook: use Colab's current numpy/transformers/peft,
# drop torchao (its version check breaks peft LoRA dispatch), install logix-ai for the LogIX arm.
!pip -q install datasets scipy scikit-learn accelerate
!pip -q uninstall -y torchao 2>/dev/null
!pip -q install --ignore-requires-python logix-ai
import numpy, transformers, peft
print('numpy', numpy.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)


Python 3.13.15
NVIDIA A100-SXM4-40GB, 40960 MiB
GPU: NVIDIA A100-SXM4-40GB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 5.1 MB/s eta 0:00:00
numpy 2.1.3 | transformers 5.20.1 | peft 0.21.2


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/traceprop_runs', exist_ok=True)


Mounted at /content/drive


In [3]:
# Repo is public; press Enter at the prompt for an anonymous clone, or paste a token.
import getpass
TOKEN = getpass.getpass('GitHub token (Enter for public clone): ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git' if TOKEN else 'https://github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!git log --oneline -1
!pip -q install -e .
%cd /content/Traceprop/experiments


GitHub token (Enter for public clone): ··········
/bin/bash: line 1: peft: command not found
fatal: Too many arguments.

usage: git clone [<options>] [--] <repo> [<dir>]

    -v, --[no-]verbose    be more verbose
    -q, --[no-]quiet      be more quiet
    --[no-]progress       force progress reporting
    --[no-]reject-shallow don't clone shallow repository
    -n, --no-checkout     don't create a checkout
    --checkout            opposite of --no-checkout
    --[no-]bare           create a bare repository
    --[no-]mirror         create a mirror repository (implies bare)
    -l, --[no-]local      to clone from a local repository
    --no-hardlinks        don't use local hardlinks, always copy
    --hardlinks           opposite of --no-hardlinks
    -s, --[no-]shared     setup as shared repository
    --[no-]recurse-submodules[=<pathspec>]
                          initialize submodules in the clone
    --[no-]recursive ...  alias of --recurse-submodules
    -j, --[no-]jobs <n>   nu

In [4]:
!mkdir -p results
!cp /content/drive/MyDrive/traceprop_runs/*.json results/ 2>/dev/null
!cp /content/drive/MyDrive/traceprop_runs/*.npz results/ 2>/dev/null
!ls results/


exp25_hf_EleutherAI_pythia-1b_track1.json
exp25_hf_EleutherAI_pythia-2.8b_track1.json
exp25_hf_EleutherAI_pythia-410m_track1.json
exp25_hf_gpt2_track1.json
exp25_pythia1b_track0_factored_kfac8.json
exp25_pythia1b_track1_factored_kfac8.json
exp25_pythia1b_track1_matched256.json
exp25_pythia1b_track6_factored_kfac8.json
exp25_pythia1b_track6_matched1536.json
exp29_hf_gpt2_rerun.json
exp29_hf_gpt2_rerun_raw.npz
exp31_logix_hf_gpt2.json
exp31_pythia1b_track0_fixed2.json
exp31_pythia1b_track0_fixed.json
exp31_pythia1b_track0.json
exp31_pythia1b_track1_fixed2.json
exp31_pythia1b_track1_fixed.json
exp31_pythia1b_track1.json
exp31_pythia1b_track6_fixed2.json
exp31_pythia1b_track6_fixed.json
exp31_pythia1b_track6.json
exp35_inlineprecond_overhead_hf_pythia7b_split.json
exp35_overhead_1b_k32.json
exp35_overhead_1b_k64.json
exp35_overhead_1b_k8.json
exp35_tiny_track0.json
exp35_tiny_track1.json
exp36_memory_storage_pythia1b.json


## Job 1 — memory/storage (item 1). A100, ~5 min.

In [5]:
!python exp36_memory_storage.py --model EleutherAI/pythia-1b --device cuda \
    --steps 200 --batch 8 --seq 128 --kfac 8 --track 0 \
    --out results/exp36_memory_storage_pythia1b.json


python3: can't open file '/content/exp36_memory_storage.py': [Errno 2] No such file or directory


## Job 2 — 6.9B logging/covariance split (item 2). A100, ~15-20 min.

Same as the saved 6.9B run but WITHOUT `--skip_log_only`, so the logging-only arm is timed.

In [6]:
!python exp35_inlineprecond_overhead_hf.py --model EleutherAI/pythia-6.9b \
    --device cuda --dtype auto --track 0 --kfac 8 --steps 50 --batch 8 --seq 128 \
    --repeats 5 --with_logix --skip_logix_pca \
    --out results/exp35_inlineprecond_overhead_hf_pythia7b_split.json


python3: can't open file '/content/exp35_inlineprecond_overhead_hf.py': [Errno 2] No such file or directory


## Job 3 — k-sweep overhead at 1B (item 8, overhead half). A100, ~30-40 min.

In [7]:
for k in [8, 32, 64]:
    print(f'=== k={k} ===')
    !python exp35_inlineprecond_overhead_hf.py --model EleutherAI/pythia-1b \
        --device cuda --dtype fp32 --track 0 --kfac {k} --steps 200 --batch 8 \
        --seq 128 --repeats 10 --with_logix \
        --out results/exp35_overhead_1b_k{k}.json


=== k=8 ===
python3: can't open file '/content/exp35_inlineprecond_overhead_hf.py': [Errno 2] No such file or directory
=== k=32 ===
python3: can't open file '/content/exp35_inlineprecond_overhead_hf.py': [Errno 2] No such file or directory
=== k=64 ===
python3: can't open file '/content/exp35_inlineprecond_overhead_hf.py': [Errno 2] No such file or directory


## Job 4 — k-sweep quality (item 8, quality half). CPU-only; runs in the same session, ~15-25 min.

In [8]:
for s in range(5):
    print(f'=== seed {s} ===')
    !python exp35_logix_lds.py --backend tiny --data synthetic --device cpu \
        --track 0 --factored --inline_precond --tune_logix --lora_init random \
        --kfac_values 8,32,64 --seed {s} \
        --out results/exp35_ksweep_seed{s}.npz


=== seed 0 ===
python3: can't open file '/content/exp35_logix_lds.py': [Errno 2] No such file or directory
=== seed 1 ===
python3: can't open file '/content/exp35_logix_lds.py': [Errno 2] No such file or directory
=== seed 2 ===
python3: can't open file '/content/exp35_logix_lds.py': [Errno 2] No such file or directory
=== seed 3 ===
python3: can't open file '/content/exp35_logix_lds.py': [Errno 2] No such file or directory
=== seed 4 ===
python3: can't open file '/content/exp35_logix_lds.py': [Errno 2] No such file or directory


## Save results to Drive

Then download these from `MyDrive/traceprop_runs/` into your local `experiments/results/`.

In [9]:
!cp results/exp36_memory_storage_pythia1b.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
!cp results/exp35_inlineprecond_overhead_hf_pythia7b_split.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
!cp results/exp35_overhead_1b_k*.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
!cp results/exp35_ksweep_seed*.npz /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
print('copied to Drive:')
!ls -la /content/drive/MyDrive/traceprop_runs/ | grep -E 'exp36|pythia7b_split|overhead_1b_k|ksweep'


copied to Drive:
-rw-------  1 root root    1497 Oct 19 21:23 exp35_inlineprecond_overhead_hf_pythia7b_split.json
-rw-------  1 root root    1967 Oct 19 21:23 exp35_overhead_1b_k32.json
-rw-------  1 root root    1968 Oct 19 21:23 exp35_overhead_1b_k64.json
-rw-------  1 root root    1958 Oct 19 21:23 exp35_overhead_1b_k8.json
-rw-------  1 root root    1212 Oct 19 21:23 exp36_memory_storage_pythia1b.json
